# Problem in the last codes
we are using batch gradient descent till now we are putting all the data in the forward pass and use all the data to make updation on the parameters which is not good in two reasons it is memory inefficient because we have to load all the data in the RAM and secondly it doesnot give better convergence.

# Solution
we divide our data into batches and apply mini batch gradient descent.

# Dataset and DataLoader class
Dataset and DataLoader are core abstractions in PyTorch that decouple how you define your data from how you efficiently iterate over it in training loops.

# whole scenario
Dataset class knows where the data is in the memory, it fetch that data row by row and DataLoader classw which is the main component divide the data into batches it decide how many rows are in each batch (batch size) and the Dataset class fetch that rows. and then these batches are send to the training loop where the training is performed.

# Dataset class
it is essentially a blueprint, when you create a custom Dataset, you decide how data is loaded and returned.

it defines:


*   **__init__()** which tells how data should be loaded.
*   **__len__()** which returns the total number of samples(rows).
*   **__getitem__(index)** which returns the data (and label) at the given index.





# DataLoader class
it wraps a dataset and handles batching, shuffling and parallel loading for you

**DataLoader control flow**

*   At the start of each epoch(if shuffles=True) shuffles indices using a sampler.
*   it divides the indices into chunk of batch sizes.
*   and each chunk is passed onto the dataset class in __ getitem __  and then it fetches that particular index row.
*   The samples are then collected and combined into a batch using collate function.
*   The batch is then returned to the training loop.







In [1]:
from sklearn.datasets import make_classification
import torch

In [2]:
X,y=make_classification(n_samples=10,
                        n_features=2,
                        n_informative=2,
                        n_redundant=0,
                        n_classes=2,
                        random_state=42)

In [3]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [4]:
X.shape

(10, 2)

In [5]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [6]:
# convert the data into pytorch tensors
X=torch.tensor(X,dtype=torch.float32)
y=torch.tensor(y,dtype=torch.long)

In [7]:
from torch.utils.data import Dataset,DataLoader

In [8]:
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features=features
    self.labels=labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    return self.features[index],self.labels[index]

In [9]:
dataset=CustomDataset(X,y)

In [10]:
len(dataset)

10

In [11]:
dataset[4]

(tensor([-1.9629, -0.9923]), tensor(0))

In [12]:
dataloader=DataLoader(dataset,batch_size=2,shuffle=True)

In [13]:
for batch_features,batch_labels in dataloader:
  print(batch_features)
  print(batch_labels)
  print('-'*50)

tensor([[ 1.7273, -1.1858],
        [-1.9629, -0.9923]])
tensor([1, 0])
--------------------------------------------------
tensor([[ 1.0683, -0.9701],
        [-0.5872, -1.9717]])
tensor([1, 0])
--------------------------------------------------
tensor([[-1.1402, -0.8388],
        [ 1.8997,  0.8344]])
tensor([0, 1])
--------------------------------------------------
tensor([[ 1.7774,  1.5116],
        [-0.7206, -0.9606]])
tensor([1, 0])
--------------------------------------------------
tensor([[-2.8954,  1.9769],
        [-0.9382, -0.5430]])
tensor([0, 1])
--------------------------------------------------



1.   You can apply any kind of transformation before returning in __ getitem __ method of CustomDataset class.
2.   The process of forming batches is sequential, we can achieve prallelization by defining num_workers in DataLoader class.


# A note about samplers
in pytorch, the sampler in the DataLoader determines the strategy for selecting samples from the dataset during data loading. It controls how indices of the dataset are drawn for each batch.

## **Types of samplers**


### **1.   Sequential sampler**  
smaples element sequentially, in the order they appear in the dataset. It is the default whn shuffles=False.
### **2.   Random sampler**
samples element randomly without replacement. Default when shuffles= True.


